In [30]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm import tqdm
from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from pathlib import Path
import random
import os
import re
import shutil

PROJECT_ROOT = Path.home() / "Desktop/Concordia/Winter 2026/COMP 472/Project"
RAW_ROOT = PROJECT_ROOT / "raw_data/stanford-cars-dataset"
DEVKIT = RAW_ROOT / "car_devkit/devkit"

CARS_TRAIN_DIR = RAW_ROOT / "cars_train"
TRAIN_MAT = DEVKIT / "cars_train_annos.mat"
META_MAT  = DEVKIT / "cars_meta.mat"

OUT_ROOT = PROJECT_ROOT / "processed_data/stanford_cars"
TRAIN_OUT = OUT_ROOT / "train"
SAMPLES_OUT = PROJECT_ROOT / "processed_data/samples/stanford_cars_train"

IMG_SIZE = 224
SEED = 42

# Create output folders
TRAIN_OUT.mkdir(parents=True, exist_ok=True)
SAMPLES_OUT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_ROOT:", RAW_ROOT)
print("DEVKIT:", DEVKIT)
print("CARS_TRAIN_DIR:", CARS_TRAIN_DIR)
print("TRAIN_MAT exists:", TRAIN_MAT.exists())
print("META_MAT exists:", META_MAT.exists())

PROJECT_ROOT: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project
RAW_ROOT: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/raw_data/stanford-cars-dataset
DEVKIT: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/raw_data/stanford-cars-dataset/car_devkit/devkit
CARS_TRAIN_DIR: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/raw_data/stanford-cars-dataset/cars_train
TRAIN_MAT exists: True
META_MAT exists: True


Helper methods

In [4]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

def is_image_file(p: Path) -> bool:
    return p.suffix.lower() in IMG_EXTS

def pil_open_rgb(path: Path) -> Image.Image:
    # open + apply EXIF rotation + force RGB (fixes 1-channel & RGBA)
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im)
        if im.mode != "RGB":
            im = im.convert("RGB")
        return im

def resize_with_padding(im: Image.Image, size: int) -> Image.Image:
    # keep aspect ratio, then pad to (size,size)
    im = ImageOps.contain(im, (size, size), method=Image.BICUBIC)
    delta_w = size - im.size[0]
    delta_h = size - im.size[1]
    padding = (delta_w // 2, delta_h // 2, delta_w - delta_w // 2, delta_h - delta_h // 2)
    return ImageOps.expand(im, padding, fill=(0, 0, 0))

def safe_filename(name: str) -> str:
    # clean folder names for Windows/macOS
    return "".join(c if c.isalnum() or c in (" ", "_", "-", ".") else "_" for c in name).strip()

def find_train_images_root(base: Path) -> Path:
    # If base has images directly
    if any(is_image_file(p) for p in base.iterdir() if p.is_file()):
        return base
    # If base has a single subfolder that contains images
    for sub in base.iterdir():
        if sub.is_dir() and any(is_image_file(p) for p in sub.iterdir() if p.is_file()):
            return sub
    # Fallback: search deeper
    jpgs = list(base.rglob("*.jpg"))
    if jpgs:
        return jpgs[0].parent
    return base

set_seed(SEED)

Load Annotations + Class Names (Train only)

In [8]:
assert TRAIN_MAT.exists(), f"Missing: {TRAIN_MAT}"
assert META_MAT.exists(), f"Missing: {META_MAT}"

# Auto-detect the actual folder that contains the training images
TRAIN_IMAGES_ROOT = find_train_images_root(CARS_TRAIN_DIR)
print("Detected TRAIN_IMAGES_ROOT:", TRAIN_IMAGES_ROOT)

train_data = loadmat(TRAIN_MAT)
meta_data  = loadmat(META_MAT)

# class names from cars_meta.mat
# Usually meta_data["class_names"] is shape (196, 1) with strings in nested arrays
class_names_raw = meta_data.get("class_names", None)
assert class_names_raw is not None, "Could not find 'class_names' in cars_meta.mat"

class_names = []
if class_names_raw.shape[0] == 1:
    iterable = class_names_raw[0]      # shape (196,)
else:
    iterable = class_names_raw[:, 0]   # shape (196,)

for val in iterable:
    if isinstance(val, np.ndarray):
        val = val[0]
    class_names.append(str(val))

print("Num classes:", len(class_names))
print("Example class:", class_names[0])

# annotations
ann = train_data.get("annotations", None)
assert ann is not None, "Could not find 'annotations' in cars_train_annos.mat"
ann = ann[0]  # (1, N) -> (N,)
print("Num train annotations:", len(ann))

Detected TRAIN_IMAGES_ROOT: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/raw_data/stanford-cars-dataset/cars_train/cars_train
Num classes: 196
Example class: AM General Hummer SUV 2000
Num train annotations: 8144


Build Manifest (image --> label)

In [9]:
rows = []
missing = 0

for a in ann:
    names = a.dtype.names  # ('bbox_x1','bbox_y1','bbox_x2','bbox_y2','class','fname')
    # fname
    fname = a["fname"][0]
    if isinstance(fname, np.ndarray):
        fname = fname[0]
    fname = str(fname)

    # class (1-based)
    class_id = int(a["class"][0][0])
    class_name = class_names[class_id - 1]  # convert to 0-based index

    img_path = TRAIN_IMAGES_ROOT / fname
    if not img_path.exists():
        missing += 1

    rows.append({
        "fname": fname,
        "img_path": str(img_path),
        "class_id": class_id,
        "class_name": class_name
    })

df_train = pd.DataFrame(rows)
print("Rows:", len(df_train))
print("Missing files:", missing)
df_train.head()

Rows: 8144
Missing files: 0


,fname,img_path,class_id,class_name
0,00001.jpg,/Users/pegahaghili/Desktop/Concordia/Winter 20...,14,Audi TTS Coupe 2012
1,00002.jpg,/Users/pegahaghili/Desktop/Concordia/Winter 20...,3,Acura TL Sedan 2012
2,00003.jpg,/Users/pegahaghili/Desktop/Concordia/Winter 20...,91,Dodge Dakota Club Cab 2007
3,00004.jpg,/Users/pegahaghili/Desktop/Concordia/Winter 20...,134,Hyundai Sonata Hybrid Sedan 2012
4,00005.jpg,/Users/pegahaghili/Desktop/Concordia/Winter 20...,106,Ford F-450 Super Duty Crew Cab 2012


Add brand extraction

In [11]:
# Common multi-word manufacturers in Stanford Cars
MULTIWORD_BRANDS = {
    "AM General",
    "Aston Martin",
    "Land Rover",
    "Mercedes-Benz",
    "Rolls-Royce",
    "Alfa Romeo",
}

def normalize_brand(b: str) -> str:
    # lower, trim, collapse spaces
    b = b.strip().lower()
    b = re.sub(r"\s+", " ", b)
    return b

def extract_brand_from_classname(class_name: str) -> str:
    """
    Input example: 'AM General Hummer SUV 2000'
    Output: 'am general'  (manufacturer)
    """
    class_name = class_name.strip()

    # If the class name starts with a known multi-word brand, use it
    for mw in sorted(MULTIWORD_BRANDS, key=len, reverse=True):
        if class_name.startswith(mw):
            return normalize_brand(mw)

    # Otherwise brand is the first token (e.g., 'Toyota', 'BMW', 'Audi', etc.)
    first = class_name.split()[0]
    return normalize_brand(first)

Build a BRAND column + BRAND→ID mapping

In [12]:
# Add brand column
df_train["brand"] = df_train["class_name"].apply(extract_brand_from_classname)

# Create stable brand->id mapping (alphabetical = reproducible)
brands_sorted = sorted(df_train["brand"].unique())
brand_to_id = {b: i for i, b in enumerate(brands_sorted)}
id_to_brand = {i: b for b, i in brand_to_id.items()}

# Add brand_id column
df_train["brand_id"] = df_train["brand"].map(brand_to_id)

# Save mapping for your project (important deliverable)
brand_map_df = pd.DataFrame({
    "brand_id": list(id_to_brand.keys()),
    "brand": [id_to_brand[i] for i in id_to_brand]
}).sort_values("brand_id")

brand_map_path = OUT_ROOT / "brand_map.csv"
brand_map_df.to_csv(brand_map_path, index=False)

print("Num unique brands:", len(brands_sorted))
print("Saved brand map:", brand_map_path)
brand_map_df.head(15)

Num unique brands: 49
Saved brand map: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/stanford_cars/brand_map.csv


,brand_id,brand
0,0,acura
1,1,am general
2,2,aston martin
3,3,audi
4,4,bentley
5,5,bmw
6,6,bugatti
7,7,buick
8,8,cadillac
9,9,chevrolet


Clean + Save Processed Train Images

In [26]:
bad_files = []
records = []

IMAGES_OUT = OUT_ROOT / "images_processed"
IMAGES_OUT.mkdir(parents=True, exist_ok=True)

for r in tqdm(df_train.itertuples(index=False),
              total=len(df_train),
              desc="Cleaning Stanford Cars (FULL dataset from train)"):
    src = Path(r.img_path)
    if not src.exists():
        bad_files.append({"file": str(src), "error": "File not found"})
        continue

    try:
        im = pil_open_rgb(src)
        im = resize_with_padding(im, IMG_SIZE)

        # Brand folder
        brand_folder = safe_filename(r.brand)
        out_dir = IMAGES_OUT / brand_folder
        out_dir.mkdir(parents=True, exist_ok=True)

        out_path = out_dir / (Path(r.fname).stem + ".jpg")
        im.save(out_path, format="JPEG", quality=95, optimize=True)

        # Save relative path
        rel_path = str(out_path.relative_to(PROJECT_ROOT)).replace("\\", "/")

        records.append({
            "path": rel_path,
            "brand": r.brand,
            "brand_id": int(r.brand_id),
        })

    except Exception as e:
        bad_files.append({"file": str(src), "error": str(e)})

# Save manifest + bad files
manifest = pd.DataFrame(records)
bad_df = pd.DataFrame(bad_files)

print("Processed:", len(manifest))
print("Bad files:", len(bad_df))

manifest_path = OUT_ROOT / "manifest.csv"
manifest.to_csv(manifest_path, index=False)

if len(bad_df) > 0:
    bad_df.to_csv(OUT_ROOT / "bad_files.csv", index=False)

print("Saved manifest:", manifest_path)

#  Save class map (brand -> brand_id)

class_map = (
    manifest[["brand", "brand_id"]]
    .drop_duplicates()
    .sort_values("brand_id")
    .reset_index(drop=True)
)

class_map_path = OUT_ROOT / "class_map.csv"
class_map.to_csv(class_map_path, index=False)

print("Saved class map:", class_map_path)

manifest.head()

Cleaning Stanford Cars (FULL dataset from train): 100%|██████████| 8144/8144 [02:00<00:00, 67.64it/s] 

Processed: 8144
Bad files: 0
Saved manifest: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/stanford_cars/manifest.csv
Saved class map: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/stanford_cars/class_map.csv


,path,brand,brand_id
0,processed_data/stanford_cars/images_processed/...,audi,3
1,processed_data/stanford_cars/images_processed/...,acura,0
2,processed_data/stanford_cars/images_processed/...,dodge,12
3,processed_data/stanford_cars/images_processed/...,hyundai,22
4,processed_data/stanford_cars/images_processed/...,ford,17


Class Counts + Stats

In [28]:
brand_counts = (
    manifest.groupby(["brand_id", "brand"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

brand_counts_path = OUT_ROOT / "brand_counts.csv"
brand_counts.to_csv(brand_counts_path, index=False)

stats_lines = [
    "Stanford Cars (FULL dataset created from official train split) - Preprocessing Stats",
    f"Total processed images: {len(manifest)}",
    f"Unique brands: {manifest['brand'].nunique()}",
    f"Target image size: {IMG_SIZE}x{IMG_SIZE}",
    f"Bad/corrupt/missing images removed: {len(bad_df)}",
    f"Brand map saved at: {brand_map_path}",
]
(OUT_ROOT / "stats_full.txt").write_text("\n".join(stats_lines), encoding="utf-8")

print("\n".join(stats_lines))
brand_counts.head(49)

Stanford Cars (FULL dataset created from official train split) - Preprocessing Stats
Total processed images: 8144
Unique brands: 49
Target image size: 224x224
Bad/corrupt/missing images removed: 0
Brand map saved at: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/stanford_cars/brand_map.csv


,brand_id,brand,count
9,9,chevrolet,905
12,12,dodge,630
3,3,audi,589
5,5,bmw,531
17,17,ford,521
22,22,hyundai,438
33,33,mercedes-benz,261
10,10,chrysler,260
0,0,acura,242
19,19,gmc,238


Save 3 Sample Images

In [29]:
sample_paths = manifest["path"].sample(n=min(3, len(manifest)), random_state=SEED).tolist()
for i, p in enumerate(sample_paths, start=1):
    p = Path(p)
    shutil.copy2(p, SAMPLES_OUT / f"sample_{i}{p.suffix.lower()}")

print("Saved samples to:", SAMPLES_OUT)
sample_paths

Saved samples to: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/samples/stanford_cars_train


['processed_data/stanford_cars/images_processed/plymouth/07438.jpg',
 'processed_data/stanford_cars/images_processed/honda/00484.jpg',
 'processed_data/stanford_cars/images_processed/aston martin/07599.jpg']

Splitting the data train/validation/test (80/10/10)


In [31]:
# Load full manifest
manifest_full = pd.read_csv(OUT_ROOT / "manifest.csv")

print("Total images:", len(manifest_full))
print("Unique brands:", manifest_full["brand"].nunique())

# Train (80%) vs Temp (20%)
train_df, temp_df = train_test_split(
    manifest_full,
    test_size=0.20,
    stratify=manifest_full["brand"],
    random_state=42
)


# Val (10%) vs Test (10%) (Split the 20% temp into two equal halves)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,   # half of 20%
    stratify=temp_df["brand"],
    random_state=42
)

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))

# Save split manifests
train_path = OUT_ROOT / "manifest_train.csv"
val_path   = OUT_ROOT / "manifest_val.csv"
test_path  = OUT_ROOT / "manifest_test.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print("Saved:")
print(" -", train_path)
print(" -", val_path)
print(" -", test_path)

# Optional sanity check: class balance
def show_distribution(df, name):
    print(f"\n{name} brand distribution (top 10):")
    display(df["brand"].value_counts().head(10))

show_distribution(train_df, "Train")
show_distribution(val_df, "Val")
show_distribution(test_df, "Test")

Total images: 8144
Unique brands: 49
Train size: 6515
Val size: 814
Test size: 815
Saved:
 - /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/stanford_cars/manifest_train.csv
 - /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/stanford_cars/manifest_val.csv
 - /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/stanford_cars/manifest_test.csv

Train brand distribution (top 10):


brand
chevrolet        724
dodge            504
audi             471
bmw              425
ford             417
hyundai          350
mercedes-benz    209
chrysler         208
acura            193
gmc              190
Name: count, dtype: int64


Val brand distribution (top 10):


brand
chevrolet        90
dodge            63
audi             59
bmw              53
ford             52
hyundai          44
mercedes-benz    26
chrysler         26
gmc              24
bentley          24
Name: count, dtype: int64


Test brand distribution (top 10):


brand
chevrolet        91
dodge            63
audi             59
bmw              53
ford             52
hyundai          44
chrysler         26
mercedes-benz    26
acura            25
gmc              24
Name: count, dtype: int64